In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
# --- Konfiguration ---
volume_path = "/Volumes/workspace/default/volume"
source_data_path = f"{volume_path}/enrollments_raw" # Eigener Ordner für die Quelldateien

# Pfade für die Checkpoints jeder Schicht
chk_bronze = f"{volume_path}/_chk/medallion_bronze"
chk_silver = f"{volume_path}/_chk/medallion_silver"
chk_gold = f"{volume_path}/_chk/medallion_gold"

In [0]:
# --- Aufräumen ---
# 1. Tabellen löschen
spark.sql("DROP TABLE IF EXISTS students")
spark.sql("DROP TABLE IF EXISTS enrollments")
spark.sql("DROP TABLE IF EXISTS enrollments_bronze")
spark.sql("DROP TABLE IF EXISTS enrollments_silver")
spark.sql("DROP TABLE IF EXISTS daily_student_courses")
print("✅ Alte Tabellen gelöscht.")

# 2. Checkpoints und Quelldaten-Ordner löschen
dbutils.fs.rm(source_data_path, recurse=True)
dbutils.fs.rm(chk_bronze, recurse=True)
dbutils.fs.rm(chk_silver, recurse=True)
dbutils.fs.rm(chk_gold, recurse=True)
print("✅ Alte Checkpoints und Quelldaten gelöscht.")

# Ausgangssituation erstellen

In [0]:
%python
dataset_school = "/Volumes/workspace/default/volume"

all_files = dbutils.fs.ls(dataset_school)
json_files = [f for f in all_files if f.name.endswith(".json")]

display(json_files)

In [0]:
# 1. JSON lesen mit Schema-Inferenz
# 'multiLine=True' hinzufügen, falls das JSON ein klassisches Array [...] ist
enrollments_df = (spark.read
    .format("json")
    .option("inferSchema", "true") 
    .load("/Volumes/workspace/default/volume/enrollments.json"))

# 2. Registrierung als temporäre View für SQL-Abfragen
enrollments_df.createOrReplaceTempView("enrollments")

# 3. Kurzer Check des Schemas
enrollments_df.printSchema()

In [0]:
# DataFrame als neue JSON-Dateien in den überwachten Volume-Pfad schreiben.
# Wichtig: Spark erstellt hier ein Verzeichnis mit Part-Dateien (z.B. part-0000...).
(enrollments_df.write
               .mode("overwrite")
               .format("json")
               .save(source_data_path)) # Wir nutzen die Variable 'source_data_path' von vorhin
               
print(f"✅ Quelldaten wurden in den Auto Loader-Ordner '{source_data_path}' geschrieben.")
print("\n🚀 System ist bereit für den ersten Durchlauf der Medallion-Pipeline.")

In [0]:
# Prüfen ob die Ausgangslage gegeben ist:
# 1. Lese die JSON-Dateien aus dem Ordner in einen neuen DataFrame
check_df = spark.read.format("json").load(source_data_path)

# 2. Zeige den Inhalt des DataFrames direkt mit display() an
print("Inhalt des Quellordners für den Auto Loader:")
display(check_df)

Nun benötigen wir noch Students

In [0]:
import pyspark.sql.functions as F

# --- Statische Lookup-Tabelle ---
# Wir lesen die Daten aus der bereitgestellten students.json
students_lookup_df = spark.read.json(f"{volume_path}/students.json")

print("✅ Studenten-Lookup-Tabelle aus 'students.json' erstellt.")
# Optional: Anzeigen, um die geladenen Daten zu prüfen
# display(students_lookup_df) 

# Bronze-Schicht – Inkrementelles Laden mit Auto Loader



Diese Zelle liest die JSON-Dateien aus dem Quellordner und schreibt sie in die enrollments_bronze-Tabelle.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col # Wichtig für den Zugriff auf die _metadata-Spalte

# Bronze-Stream: Rohdaten aufnehmen + Metadaten hinzufügen
(spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaLocation", f"{chk_bronze}/_schema") # Schema-Separation
      .load(source_data_path)
      .select(
          "*", 
          F.current_timestamp().alias("arrival_time"), # Wann kamen die Daten an?
          col("_metadata.file_path").alias("source_file") # Aus welcher Datei?
      )
 .writeStream
      .format("delta")
      .option("checkpointLocation", chk_bronze)
      .outputMode("append")
      .trigger(availableNow=True) # Wichtig für Free Edition
      .table("workspace.default.enrollments_bronze") # Unity Catalog Pfad
      .awaitTermination() # Warten, bis dieser Schritt fertig ist
)

print("✅ Bronze-Schicht erfolgreich verarbeitet.")
display(spark.table("workspace.default.enrollments_bronze"))

# Silber-Schicht – Bereinigen und Anreichern

Diese Zelle liest aus der Bronze-Tabelle, filtert die Daten und reichert sie durch einen Join mit statischen Studentendaten an.

In [0]:
import pyspark.sql.functions as F

# Silber-Stream: Liest von Bronze, bereinigt und reichert an
enrollments_enriched_df = (
    spark.readStream
         .table("workspace.default.enrollments_bronze") # Konsistente Benennung
         .where("quantity > 0")                   # Datenbereinigung
         # Umwandlung von String/Long in Timestamp für Zeit-Analysen
         .withColumn("processed_at", F.to_timestamp("timestamp")) 
         # JOIN mit statischen Stammdaten (Stream-Static Join)
         .join(students_lookup_df, "student_id", "left") # Left-Join ist oft sicherer
         .select(
             "enroll_id", 
             "student_id", 
             "email", 
             "gpa", 
             "profile", 
             "quantity", 
             "courses", 
             "processed_at"
         )
)

# Ergebnis in die Silber-Tabelle schreiben
(enrollments_enriched_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", chk_silver)
                        .option("mergeSchema", "true")
                        .outputMode("append")
                        .trigger(availableNow=True) # Kosteneffizient in der Free Edition
                        .table("workspace.default.enrollments_silver")
                        .awaitTermination()
)

print("✅ Silber-Schicht erfolgreich verarbeitet (Joined & Filtered).")
display(spark.table("workspace.default.enrollments_silver"))

# Gold-Schicht – Aggregation für Business-Insights

Diese Zelle liest aus der Silber-Tabelle und erstellt eine aggregierte Sicht für das Reporting.

In [0]:
import pyspark.sql.functions as F

# Gold-Stream: Liest von Silber und erstellt tägliche Zusammenfassungen
enrollments_agg_df = (
    spark.readStream
         .table("workspace.default.enrollments_silver")
         # Wir runden den Zeitstempel auf den Tag ab (DD)
         .withColumn("day", F.date_trunc("DD", "processed_at")) 
         .groupBy("student_id", "email", "day")
         .agg(F.sum("quantity").alias("total_courses_enrolled"))
)

# Ergebnis in die Gold-Tabelle schreiben
(enrollments_agg_df.writeStream
                   .format("delta")
                   .outputMode("complete") # Zwingend bei Aggregationen ohne Watermarking
                   .option("checkpointLocation", chk_gold)
                   .trigger(availableNow=True) # Perfekt für die Free Edition
                   .table("workspace.default.daily_student_courses")
                   .awaitTermination()
)

print("✅ Gold-Schicht erfolgreich verarbeitet (Aggregiert & Report-bereit).")
display(spark.table("workspace.default.daily_student_courses"))

# Neue Daten simulieren

Jetzt simulieren wir, dass ein neuer Tag anbricht und neue Daten eintreffen.



In [0]:
# Inhalt für eine zweite Datei
json_content_part2 = """
{"enroll_id": "e004", "timestamp": "2025-08-04T11:30:00", "student_id": 1, "quantity": 1, "total": 600, "courses": [103]}
{"enroll_id": "e005", "timestamp": "2025-08-05T11:30:00", "student_id": 2, "quantity": 1, "total": 850, "courses": [101]}
"""

# Die neue Datei in den Quellordner schreiben
dbutils.fs.put(f"{source_data_path}/enrollments_part2.json", json_content_part2, overwrite=True)

print("✅ Neue Datendatei 'enrollments_part2.json' wurde mit korrektem Schema hinzugefügt.")

# Pipeline erneut ausführen und inkrementellen Lauf beobachten

Um die neuen Daten durch die Pipeline zu bewegen, führe einfach die Bronze, Silver und Gold-Schicht erneut in dieser Reihenfolge aus. Jede Schicht wird dank der Checkpoints nur die neuen Daten verarbeiten. Sie werden sehen, wie sich die Tabellen am Ende aktualisieren.

# Aufräumen

In [0]:
%sql
-- ACHTUNG: WIR BRAUCHEN DIE TABELLEN NOCH!!!!!

-- DROP TABLE IF EXISTS daily_student_courses;
-- DROP TABLE IF EXISTS enrollments_bronze;
-- DROP TABLE IF EXISTS enrollments_silver;